In [1]:
# ============================================================================
# COMPLETE SPACE EFFICIENCY EVALUATION FOR MOE PHISHING DETECTION SYSTEM
# ============================================================================

import warnings
warnings.filterwarnings('ignore')

import sys
import torch
import psutil
import gc
import numpy as np
import pickle
import io
import tempfile
import os
import shutil
import re
from urllib.parse import urlparse

# ============================================================================
# URL FEATURES CLASS (REQUIRED FOR UNPICKLING)
# ============================================================================

class URLFeatures:
    """Feature extractor for URL analysis - required for unpickling the model"""
    
    def __init__(self):
        pass
    
    def extract_features(self, url):
        """Extract features from a URL"""
        try:
            parsed = urlparse(url)
            
            features = {
                'url_length': len(url),
                'domain_length': len(parsed.netloc),
                'path_length': len(parsed.path),
                'num_dots': url.count('.'),
                'num_hyphens': url.count('-'),
                'num_underscores': url.count('_'),
                'num_slashes': url.count('/'),
                'num_digits': sum(c.isdigit() for c in url),
                'has_ip': self._has_ip_address(parsed.netloc),
                'has_https': 1 if parsed.scheme == 'https' else 0,
                'num_subdomains': len(parsed.netloc.split('.')) - 2 if parsed.netloc else 0,
                'has_at_symbol': 1 if '@' in url else 0,
                'has_double_slash': 1 if '//' in parsed.path else 0,
                'num_special_chars': len(re.findall(r'[!@#$%^&*()+=\[\]{};:\'",<>?\\|`~]', url)),
            }
            
            return list(features.values())
        except Exception as e:
            # Return default features on error
            return [0] * 15
    
    def _has_ip_address(self, domain):
        """Check if domain is an IP address"""
        ip_pattern = r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b'
        return 1 if re.match(ip_pattern, domain) else 0

# ============================================================================
# IMPROVED MEMORY MEASUREMENT FUNCTIONS (WINDOWS-COMPATIBLE)
# ============================================================================

def get_sklearn_model_size(model, name="Model"):
    """Accurate size for sklearn models using in-memory serialization"""
    try:
        buffer = io.BytesIO()
        pickle.dump(model, buffer)
        size_bytes = buffer.tell()
        buffer.close()
        
        return {
            'name': name,
            'size_mb': size_bytes / (1024**2),
            'size_kb': size_bytes / 1024,
            'size_bytes': size_bytes
        }
    except Exception as e:
        print(f"  WARNING: Could not measure {name}: {e}")
        return {'name': name, 'size_mb': 0, 'size_kb': 0, 'size_bytes': 0}

def get_torch_model_size(model, name="Model"):
    """Accurate size for PyTorch models"""
    try:
        param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
        buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
        total_size_bytes = param_size + buffer_size
        
        return {
            'name': name,
            'parameters_mb': param_size / (1024**2),
            'buffers_mb': buffer_size / (1024**2),
            'total_mb': total_size_bytes / (1024**2),
            'total_gb': total_size_bytes / (1024**3),
            'size_bytes': total_size_bytes
        }
    except Exception as e:
        print(f"  WARNING: Could not measure {name}: {e}")
        return {'name': name, 'total_mb': 0, 'total_gb': 0, 'size_bytes': 0, 'parameters_mb': 0, 'buffers_mb': 0}

def get_tokenizer_size(tokenizer, name="Tokenizer"):
    """Accurate size for HuggingFace tokenizer"""
    try:
        tmp_dir = tempfile.mkdtemp()
        try:
            tokenizer.save_pretrained(tmp_dir)
            total_size = sum(
                os.path.getsize(os.path.join(root, file))
                for root, dirs, files in os.walk(tmp_dir)
                for file in files
            )
            return {
                'name': name,
                'size_mb': total_size / (1024**2),
                'size_kb': total_size / 1024,
                'size_bytes': total_size,
                'vocab_size': len(tokenizer)
            }
        finally:
            try:
                shutil.rmtree(tmp_dir)
            except:
                pass
    except Exception as e:
        print(f"  WARNING: Could not measure tokenizer: {e}")
        return {'name': name, 'size_mb': 0, 'size_kb': 0, 'size_bytes': 0, 'vocab_size': 0}

def get_system_memory():
    """Get current system memory usage"""
    try:
        process = psutil.Process()
        mem_info = process.memory_info()
        vm = psutil.virtual_memory()
        
        return {
            'rss_mb': mem_info.rss / (1024**2),
            'vms_mb': mem_info.vms / (1024**2),
            'rss_gb': mem_info.rss / (1024**3),
            'system_available_gb': vm.available / (1024**3),
            'system_total_gb': vm.total / (1024**3),
            'system_percent': vm.percent
        }
    except Exception as e:
        print(f"  WARNING: Could not get system memory: {e}")
        return {'rss_mb': 0, 'vms_mb': 0, 'rss_gb': 0, 'system_available_gb': 0, 'system_total_gb': 0, 'system_percent': 0}

def get_gpu_memory():
    """Get GPU memory usage if available"""
    try:
        if torch.cuda.is_available():
            return {
                'allocated_mb': torch.cuda.memory_allocated() / (1024**2),
                'reserved_mb': torch.cuda.memory_reserved() / (1024**2),
                'allocated_gb': torch.cuda.memory_allocated() / (1024**3),
                'reserved_gb': torch.cuda.memory_reserved() / (1024**3),
                'max_allocated_gb': torch.cuda.max_memory_allocated() / (1024**3),
            }
    except:
        pass
    return None

def estimate_list_size(lst, name="List"):
    """Estimate size of list"""
    try:
        if not lst:
            return {'name': name, 'size_mb': 0, 'size_kb': 0}
        
        base_size = sys.getsizeof(lst)
        sample_size = min(100, len(lst))
        item_size = sum(sys.getsizeof(item) for item in lst[:sample_size]) / sample_size
        total_size = base_size + (item_size * len(lst))
        
        return {
            'name': name,
            'size_mb': total_size / (1024**2),
            'size_kb': total_size / 1024
        }
    except Exception as e:
        return {'name': name, 'size_mb': 0, 'size_kb': 0}

# ============================================================================
# GATING NETWORK CLASS
# ============================================================================

class GatingNetwork(torch.nn.Module):
    def __init__(self, input_size=8, hidden_size=64, num_experts=2):
        super(GatingNetwork, self).__init__()
        self.fc1 = torch.nn.Linear(input_size, hidden_size)
        self.relu = torch.nn.ReLU()
        self.fc2 = torch.nn.Linear(hidden_size, num_experts)
        self.softmax = torch.nn.Softmax(dim=1)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        weights = self.softmax(x)
        return weights

# ============================================================================
# COMPREHENSIVE SPACE EFFICIENCY EVALUATION
# ============================================================================

def evaluate_space_efficiency():
    """Complete space efficiency evaluation of the MoE system"""
    
    print("="*80)
    print("SPACE EFFICIENCY EVALUATION - MOE PHISHING DETECTION SYSTEM")
    print("="*80)
    
    results = {
        'models': {},
        'memory': {},
        'data_structures': {},
        'recommendations': []
    }
    
    # ========================================================================
    # 1. MODEL SIZES (ACCURATE)
    # ========================================================================
    
    print("\n" + "="*80)
    print("MODEL MEMORY FOOTPRINT (ACCURATE MEASUREMENTS)")
    print("="*80)
    
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    # Expert 1 (URL Expert - sklearn)
    try:
        import joblib
        
        # Try multiple possible paths
        possible_paths = [
            r"C:\Users\angelo\Downloads\THESIS\URL_Expert-20251210T060216Z-1-001\URL_Expert\Notebook and Model\url_expert_1.pkl",
            r"C:\Users\angelo\Downloads\THESIS\URL_Expert-20241210T060216Z-1-001\URL_Expert\Notebook and Model\url_expert_1.pkl",
        ]
        
        # Search for the file
        thesis_path = r"C:\Users\angelo\Downloads\THESIS"
        if os.path.exists(thesis_path):
            for root, dirs, files in os.walk(thesis_path):
                for file in files:
                    if file == "url_expert_1.pkl":
                        possible_paths.insert(0, os.path.join(root, file))
        
        # Try each path
        expert_1 = None
        URL_MODEL_PATH = None
        for path in possible_paths:
            if os.path.exists(path):
                URL_MODEL_PATH = path
                expert_1 = joblib.load(path)
                break
        
        if expert_1 is None:
            raise FileNotFoundError("url_expert_1.pkl not found in any location")
        
        expert_1_size = get_sklearn_model_size(expert_1, "URL Expert")
        results['models']['url_expert'] = expert_1_size
        
        print(f"\n[1] URL Expert (Random Forest):")
        print(f"    Path: {URL_MODEL_PATH}")
        print(f"    Size: {expert_1_size['size_mb']:.2f} MB ({expert_1_size['size_kb']:.2f} KB)")
        print(f"    Type: Scikit-learn model (CPU)")
        
        if hasattr(expert_1, 'n_estimators'):
            print(f"    Trees: {expert_1.n_estimators}")
        if hasattr(expert_1, 'n_features_in_'):
            print(f"    Features: {expert_1.n_features_in_}")
        
    except Exception as e:
        print(f"\n[1] URL Expert: WARNING - Could not load - {e}")
        results['models']['url_expert'] = {'size_mb': 0}
    
    # Expert 2 (Text Expert - DistilBERT)
    try:
        from transformers import AutoModelForSequenceClassification
        TEXT_MODEL_PATH = r"C:\Users\angelo\Downloads\THESIS\distilbert_phishing_model"
        expert_2 = AutoModelForSequenceClassification.from_pretrained(TEXT_MODEL_PATH)
        
        expert_2_size = get_torch_model_size(expert_2, "Text Expert (DistilBERT)")
        results['models']['text_expert'] = expert_2_size
        
        print(f"\n[2] Text Expert (DistilBERT):")
        print(f"    Parameters: {expert_2_size['parameters_mb']:.2f} MB")
        print(f"    Buffers: {expert_2_size['buffers_mb']:.2f} MB")
        print(f"    Total: {expert_2_size['total_mb']:.2f} MB ({expert_2_size['total_gb']:.3f} GB)")
        print(f"    Type: Transformer model")
        
        total_params = sum(p.numel() for p in expert_2.parameters())
        trainable_params = sum(p.numel() for p in expert_2.parameters() if p.requires_grad)
        print(f"    Total parameters: {total_params:,}")
        print(f"    Trainable parameters: {trainable_params:,}")
        
    except Exception as e:
        print(f"\n[2] Text Expert: WARNING - Could not load - {e}")
        results['models']['text_expert'] = {'total_mb': 0, 'parameters_mb': 0, 'buffers_mb': 0, 'total_gb': 0}
    
    # Gating Network
    try:
        gating_net = GatingNetwork(input_size=8, hidden_size=64, num_experts=2)
        gating_net.load_state_dict(torch.load('gating_network.pth'))
        
        gating_size = get_torch_model_size(gating_net, "Gating Network")
        results['models']['gating_network'] = gating_size
        
        print(f"\n[3] Gating Network:")
        print(f"    Parameters: {gating_size['parameters_mb']:.2f} MB ({gating_size['parameters_mb']*1024:.2f} KB)")
        print(f"    Total: {gating_size['total_mb']:.2f} MB")
        print(f"    Type: Small feedforward network")
        
        total_params = sum(p.numel() for p in gating_net.parameters())
        print(f"    Total parameters: {total_params:,}")
        
    except Exception as e:
        print(f"\n[3] Gating Network: WARNING - Could not load - {e}")
        results['models']['gating_network'] = {'total_mb': 0, 'parameters_mb': 0}
    
    # Tokenizer
    try:
        from transformers import AutoTokenizer
        TEXT_MODEL_PATH = r"C:\Users\angelo\Downloads\THESIS\distilbert_phishing_model"
        tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_PATH)
        
        tokenizer_size = get_tokenizer_size(tokenizer, "Tokenizer")
        results['models']['tokenizer'] = tokenizer_size
        
        print(f"\n[4] Tokenizer:")
        print(f"    Size: {tokenizer_size['size_mb']:.2f} MB ({tokenizer_size['size_kb']:.2f} KB)")
        print(f"    Vocab size: {tokenizer_size.get('vocab_size', len(tokenizer)):,}")
        print(f"    Type: DistilBERT tokenizer")
        
    except Exception as e:
        print(f"\n[4] Tokenizer: WARNING - Could not load - {e}")
        results['models']['tokenizer'] = {'size_mb': 0}
    
    # ========================================================================
    # 2. SYSTEM MEMORY USAGE
    # ========================================================================
    
    print("\n" + "="*80)
    print("SYSTEM MEMORY USAGE")
    print("="*80)
    
    sys_mem = get_system_memory()
    results['memory']['system'] = sys_mem
    
    print(f"\n[CPU Memory]")
    print(f"  Process RSS: {sys_mem.get('rss_mb', 0):.2f} MB ({sys_mem.get('rss_gb', 0):.2f} GB)")
    print(f"  Process VMS: {sys_mem.get('vms_mb', 0):.2f} MB")
    print(f"  System Available: {sys_mem.get('system_available_gb', 0):.2f} GB / {sys_mem.get('system_total_gb', 0):.2f} GB")
    print(f"  System Usage: {sys_mem.get('system_percent', 0):.1f}%")
    
    # GPU Memory
    gpu_mem = get_gpu_memory()
    if gpu_mem:
        results['memory']['gpu'] = gpu_mem
        
        print(f"\n[GPU Memory]")
        print(f"  Allocated: {gpu_mem['allocated_mb']:.2f} MB ({gpu_mem['allocated_gb']:.3f} GB)")
        print(f"  Reserved: {gpu_mem['reserved_mb']:.2f} MB ({gpu_mem['reserved_gb']:.3f} GB)")
        print(f"  Peak Allocated: {gpu_mem['max_allocated_gb']:.3f} GB")
    else:
        print(f"\n[GPU Memory]")
        print(f"  No GPU available (CPU-only mode)")
    
    # ========================================================================
    # 3. DATA STRUCTURES MEMORY
    # ========================================================================
    
    print("\n" + "="*80)
    print("DATA STRUCTURES MEMORY")
    print("="*80)
    
    num_samples = 10000
    
    # Input data
    sample_texts = ["URGENT! Verify account"] * num_samples
    sample_urls = ["http://phishing.com"] * num_samples
    
    texts_size = estimate_list_size(sample_texts, "Input Texts (10k)")
    urls_size = estimate_list_size(sample_urls, "Input URLs (10k)")
    
    results['data_structures']['input_texts'] = texts_size
    results['data_structures']['input_urls'] = urls_size
    
    print(f"\n[Input Data - 10,000 samples]")
    print(f"  Texts: {texts_size['size_mb']:.2f} MB")
    print(f"  URLs: {urls_size['size_mb']:.2f} MB")
    print(f"  Total Input: {texts_size['size_mb'] + urls_size['size_mb']:.2f} MB")
    
    # Output data
    sample_results = {
        'predictions': ['PHISHING'] * num_samples,
        'confidences': [95.5] * num_samples,
        'url_weights': [60.0] * num_samples,
        'text_weights': [40.0] * num_samples,
    }
    
    results_size = estimate_list_size(sample_results['predictions'], "Results")
    results['data_structures']['output'] = results_size
    
    print(f"\n[Output Data - 10,000 samples]")
    print(f"  Predictions & Metadata: ~{results_size['size_mb']*4:.2f} MB")
    
    # Intermediate tensors
    print(f"\n[Intermediate Processing]")
    
    batch_size = 64
    max_length = 128
    hidden_size = 768
    
    input_ids_size = batch_size * max_length * 4 / (1024**2)
    attention_mask_size = batch_size * max_length * 4 / (1024**2)
    hidden_states_size = batch_size * max_length * hidden_size * 4 / (1024**2)
    
    print(f"  Tokenized batch (64 samples):")
    print(f"    Input IDs: {input_ids_size:.2f} MB")
    print(f"    Attention masks: {attention_mask_size:.2f} MB")
    print(f"    Hidden states: {hidden_states_size:.2f} MB")
    print(f"    Total per batch: {input_ids_size + attention_mask_size + hidden_states_size:.2f} MB")
    
    # ========================================================================
    # 4. TOTAL SPACE ANALYSIS
    # ========================================================================
    
    print("\n" + "="*80)
    print("TOTAL SPACE ANALYSIS")
    print("="*80)
    
    # Calculate total model size
    total_model_size = 0
    if 'url_expert' in results['models']:
        total_model_size += results['models']['url_expert'].get('size_mb', 0)
    if 'text_expert' in results['models']:
        total_model_size += results['models']['text_expert'].get('total_mb', 0)
    if 'gating_network' in results['models']:
        total_model_size += results['models']['gating_network'].get('total_mb', 0)
    if 'tokenizer' in results['models']:
        total_model_size += results['models']['tokenizer'].get('size_mb', 0)
    
    results['total'] = {
        'models_mb': total_model_size,
        'models_gb': total_model_size / 1024
    }
    
    print(f"\n[Storage Requirements]")
    print(f"  Total Models: {total_model_size:.2f} MB ({total_model_size/1024:.2f} GB)")
    print(f"  Breakdown:")
    print(f"    - URL Expert: {results['models'].get('url_expert', {}).get('size_mb', 0):.2f} MB")
    print(f"    - Text Expert: {results['models'].get('text_expert', {}).get('total_mb', 0):.2f} MB")
    print(f"    - Gating Network: {results['models'].get('gating_network', {}).get('total_mb', 0):.2f} MB")
    print(f"    - Tokenizer: {results['models'].get('tokenizer', {}).get('size_mb', 0):.2f} MB")
    
    print(f"\n[Runtime Memory (10k samples)]")
    runtime_memory = (
        texts_size['size_mb'] + 
        urls_size['size_mb'] + 
        results_size['size_mb'] * 4 +
        (input_ids_size + attention_mask_size + hidden_states_size)
    )
    print(f"  Data + Intermediate: ~{runtime_memory:.2f} MB")
    print(f"  Peak Memory (Models + Data): ~{total_model_size + runtime_memory:.2f} MB")
    print(f"                                ({(total_model_size + runtime_memory)/1024:.2f} GB)")
    
    # ========================================================================
    # 5. EFFICIENCY RATINGS
    # ========================================================================
    
    print("\n" + "="*80)
    print("SPACE EFFICIENCY RATINGS")
    print("="*80)
    
    text_expert_size = results['models'].get('text_expert', {}).get('total_mb', 0)
    
    print(f"\n[Model Efficiency]")
    if text_expert_size < 100:
        rating = "EXCELLENT"
        results['recommendations'].append("Model size is very efficient")
    elif text_expert_size < 500:
        rating = "GOOD"
        results['recommendations'].append("Model size is reasonable")
    elif text_expert_size < 1000:
        rating = "FAIR"
        results['recommendations'].append("Consider model quantization")
    else:
        rating = "POOR"
        results['recommendations'].append("Model is too large, quantization recommended")
    
    print(f"  Text Expert (DistilBERT): {rating}")
    print(f"    Size: {text_expert_size:.2f} MB")
    
    gating_size = results['models'].get('gating_network', {}).get('total_mb', 0)
    if gating_size < 1:
        print(f"  Gating Network: EXCELLENT (minimal overhead)")
    
    url_expert_size = results['models'].get('url_expert', {}).get('size_mb', 0)
    if url_expert_size < 50:
        print(f"  URL Expert: EXCELLENT (lightweight)")
    
    # Memory efficiency per prediction
    print(f"\n[Memory Efficiency Per Prediction]")
    mem_per_sample = runtime_memory / num_samples
    print(f"  Memory per sample: {mem_per_sample:.4f} MB ({mem_per_sample*1024:.2f} KB)")
    
    if mem_per_sample < 0.01:
        print(f"  Rating: EXCELLENT (very memory efficient)")
    elif mem_per_sample < 0.1:
        print(f"  Rating: GOOD")
    else:
        print(f"  Rating: FAIR (could be optimized)")
    
    
    # ========================================================================
    # COMPARISON WITH ALTERNATIVES
    # ========================================================================
    
    print("\n" + "="*80)
    print("COMPARISON WITH ALTERNATIVE ARCHITECTURES")
    print("="*80)
    
    print(f"\n[Model Size Comparison]")
    comparisons = {
        'Your DistilBERT': text_expert_size,
        'BERT-Base': 440,
        'RoBERTa-Base': 498,
        'ALBERT-Base': 47,
        'MobileBERT': 100,
        'TinyBERT': 57,
    }
    
    for model_name, size_mb in comparisons.items():
        if 'Your' in model_name:
            print(f"  {model_name}: {size_mb:.0f} MB <- Your model")
        else:
            if size_mb > text_expert_size:
                diff = ((size_mb - text_expert_size) / text_expert_size) * 100
                print(f"  {model_name}: {size_mb:.0f} MB (+{diff:.0f}% larger)")
            else:
                diff = ((text_expert_size - size_mb) / text_expert_size) * 100
                print(f"  {model_name}: {size_mb:.0f} MB (-{diff:.0f}% smaller)")
    
    # ========================================================================
    # FINAL SUMMARY
    # ========================================================================
    
    print("\n" + "="*80)
    print("FINAL SUMMARY")
    print("="*80)
    
    print(f"\nTOTAL STORAGE: {total_model_size:.2f} MB ({total_model_size/1024:.2f} GB)")
    print(f"RUNTIME MEMORY: ~{runtime_memory:.2f} MB for 10k samples")
    print(f"PEAK MEMORY: ~{(total_model_size + runtime_memory)/1024:.2f} GB")
    
    if total_model_size < 300:
        print(f"\nVERDICT: EXCELLENT - Your system is very space-efficient")
    elif total_model_size < 500:
        print(f"\nVERDICT: GOOD - Reasonable space requirements for production")
    else:
        print(f"\nVERDICT: FAIR - Consider optimization techniques")
    
    print("="*80)
    
    return results

# ============================================================================
# RUN EVALUATION
# ============================================================================

if __name__ == "__main__":
    results = evaluate_space_efficiency()
    
    print("\n" + "="*80)
    print("SPACE EFFICIENCY EVALUATION COMPLETE")
    print("="*80)

SPACE EFFICIENCY EVALUATION - MOE PHISHING DETECTION SYSTEM

MODEL MEMORY FOOTPRINT (ACCURATE MEASUREMENTS)

[1] URL Expert (Random Forest):
    Path: C:\Users\angelo\Downloads\THESIS\URL_Expert-20251210T060216Z-1-001\URL_Expert\Notebook and Model\url_expert_1.pkl
    Size: 0.13 MB (129.56 KB)
    Type: Scikit-learn model (CPU)
    Features: 1

[2] Text Expert (DistilBERT):
    Parameters: 255.41 MB
    Buffers: 0.00 MB
    Total: 255.42 MB (0.249 GB)
    Type: Transformer model
    Total parameters: 66,955,010
    Trainable parameters: 66,955,010

[3] Gating Network:
    Parameters: 0.00 MB (2.76 KB)
    Total: 0.00 MB
    Type: Small feedforward network
    Total parameters: 706

[4] Tokenizer:
    Size: 0.90 MB (923.22 KB)
    Vocab size: 30,522
    Type: DistilBERT tokenizer

SYSTEM MEMORY USAGE

[CPU Memory]
  Process RSS: 584.16 MB (0.57 GB)
  Process VMS: 1130.02 MB
  System Available: 3.26 GB / 15.27 GB
  System Usage: 78.6%

[GPU Memory]
  Allocated: 0.00 MB (0.000 GB)
  Reser

In [10]:
# ============================================================================
# COMPLETE SPACE EFFICIENCY & RESOURCE USAGE EVALUATION
# MOE Phishing Detection System (FULLY FIXED + CPU PEAK + RAM MB)
# ============================================================================

import warnings
warnings.filterwarnings("ignore")

import os
import io
import gc
import json
import time
import psutil
import pickle
import shutil
import tempfile
from datetime import datetime
from pathlib import Path

import torch
import numpy as np

# ============================================================================
# REQUIRED FOR URL EXPERT DESERIALIZATION
# ============================================================================

class URLFeatures:
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X


# ============================================================================
# CONFIGURATION
# ============================================================================

class EvaluationConfig:
    URL_MODEL_PATH = r"C:\Users\angelo\Downloads\THESIS\URL_Expert-20251210T060216Z-1-001\URL_Expert\Notebook and Model\url_expert_1.pkl"
    TEXT_MODEL_PATH = r"C:\Users\angelo\Downloads\THESIS\distilbert_phishing_model"
    GATING_NETWORK_PATH = "gating_network.pth"

    MAX_SEQUENCE_LENGTH = 128
    CPU_SAMPLE_INTERVAL = 0.1   # seconds


# ============================================================================
# RESOURCE MONITOR
# ============================================================================

class ResourceMonitor:
    def __init__(self, interval=0.1):
        self.interval = interval
        self.cpu_peaks = []
        self.running = False

    def start(self):
        self.running = True
        psutil.cpu_percent(interval=None)
        while self.running:
            self.cpu_peaks.append(psutil.cpu_percent(interval=self.interval))

    def stop(self):
        self.running = False

    def peak_cpu(self):
        return max(self.cpu_peaks) if self.cpu_peaks else 0.0


# ============================================================================
# MEMORY PROFILER
# ============================================================================

class MemoryProfiler:

    @staticmethod
    def sklearn_model_size(model):
        buffer = io.BytesIO()
        pickle.dump(model, buffer)
        size_mb = buffer.tell() / (1024 ** 2)
        buffer.close()
        return size_mb

    @staticmethod
    def torch_model_size(model):
        param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
        return {
            "parameters": sum(p.numel() for p in model.parameters()),
            "size_mb": param_bytes / (1024 ** 2)
        }

    @staticmethod
    def tokenizer_size(tokenizer):
        tmp = tempfile.mkdtemp()
        tokenizer.save_pretrained(tmp)
        size = sum(
            os.path.getsize(os.path.join(root, f))
            for root, _, files in os.walk(tmp)
            for f in files
        )
        shutil.rmtree(tmp)
        return size / (1024 ** 2)

    @staticmethod
    def process_memory():
        p = psutil.Process()
        m = p.memory_info()
        v = psutil.virtual_memory()
        return {
            "process_rss_mb": m.rss / (1024 ** 2),
            "system_total_mb": v.total / (1024 ** 2),
            "system_used_mb": v.used / (1024 ** 2),
            "system_available_mb": v.available / (1024 ** 2),
            "system_used_percent": v.percent
        }


# ============================================================================
# GATING NETWORK (MUST MATCH CHECKPOINT)
# ============================================================================

class GatingNetwork(torch.nn.Module):
    def __init__(self, input_size=8, hidden_size=64, num_experts=2):
        super().__init__()
        self.fc1 = torch.nn.Linear(input_size, hidden_size)
        self.relu = torch.nn.ReLU()
        self.fc2 = torch.nn.Linear(hidden_size, num_experts)
        self.softmax = torch.nn.Softmax(dim=1)

    def forward(self, x):
        return self.softmax(self.fc2(self.relu(self.fc1(x))))


# ============================================================================
# MODEL MANAGER
# ============================================================================

class ModelManager:
    def __init__(self, config):
        self.config = config
        self.models = {}
        self.stats = {}

    def load_models(self):
        import joblib
        from transformers import AutoTokenizer, AutoModelForSequenceClassification

        # URL Expert
        url_model = joblib.load(self.config.URL_MODEL_PATH)
        self.models["url_expert"] = url_model
        self.stats["url_expert_mb"] = MemoryProfiler.sklearn_model_size(url_model)

        # Text Expert
        tokenizer = AutoTokenizer.from_pretrained(self.config.TEXT_MODEL_PATH)
        text_model = AutoModelForSequenceClassification.from_pretrained(
            self.config.TEXT_MODEL_PATH
        )
        text_model.eval()

        self.models["tokenizer"] = tokenizer
        self.models["text_expert"] = text_model

        self.stats["text_expert"] = MemoryProfiler.torch_model_size(text_model)
        self.stats["tokenizer_mb"] = MemoryProfiler.tokenizer_size(tokenizer)

        # Gating Network
        gate = GatingNetwork()
        state = torch.load(self.config.GATING_NETWORK_PATH, map_location="cpu")
        gate.load_state_dict(state, strict=True)
        gate.eval()

        self.models["gating_network"] = gate
        self.stats["gating_network"] = MemoryProfiler.torch_model_size(gate)

        return self.models, self.stats


# ============================================================================
# EVALUATOR
# ============================================================================

class SpaceEfficiencyEvaluator:
    def __init__(self, config):
        self.config = config
        self.results = {
            "timestamp": datetime.now().isoformat(),
            "models": {},
            "memory": {},
            "cpu": {}
        }

    def run(self):
        monitor = ResourceMonitor(self.config.CPU_SAMPLE_INTERVAL)
        import threading

        t = threading.Thread(target=monitor.start)
        t.start()

        manager = ModelManager(self.config)
        _, model_stats = manager.load_models()

        time.sleep(0.5)
        monitor.stop()
        t.join()

        self.results["models"] = model_stats
        self.results["memory"] = MemoryProfiler.process_memory()
        self.results["cpu"]["peak_percent"] = monitor.peak_cpu()

        return self.results

    def save(self, filename="space_efficiency_results.json"):
        with open(filename, "w") as f:
            json.dump(self.results, f, indent=2)


# ============================================================================
# MAIN
# ============================================================================

def main():
    config = EvaluationConfig()
    evaluator = SpaceEfficiencyEvaluator(config)
    results = evaluator.run()
    evaluator.save()

    print("\nRESOURCE SUMMARY")
    print(f"CPU Peak Usage: {results['cpu']['peak_percent']:.2f}%")
    print(f"Process RAM (RSS): {results['memory']['process_rss_mb']:.2f} MB")

    return results


if __name__ == "__main__":
    main()



RESOURCE SUMMARY
CPU Peak Usage: 49.40%
Process RAM (RSS): 286.07 MB
